# Testing agent recovery from tool failures with the OpenAI Agents SDK

This notebook builds a delivery-support agent and tests how it responds when its tools time out, return transient errors, produce invalid data, or leave a write in an ambiguous state.

## What you'll build

- A deterministic tool-failure simulator.
- A typed recovery layer with bounded retries and output validation.
- An idempotent write path that prevents duplicate side effects.
- A single [OpenAI Agents SDK](https://developers.openai.com/api/docs/guides/agents) agent that uses structured tool outcomes.
- A repeatable scenario matrix that checks recovery, fallback, and escalation behavior.

The fault simulator and recovery assertions run locally. The final agent run is opt-in because it calls the OpenAI API.

## Prerequisites

You need a Python environment that supports the OpenAI Agents SDK. To run the opt-in agent section, set `OPENAI_API_KEY` in your environment. You can override the example model with `OPENAI_MODEL` and enable live calls with `RUN_LIVE_AGENT=true`.

The next cell installs the only third-party packages used in the notebook:

- `openai-agents` for the agent loop, function tools, and tracing.
- `pydantic` for explicit tool input and output schemas.
- `pandas` for the final scenario-results table.

In [ ]:
%pip install -q openai-agents pydantic pandas

## Configure the notebook

Offline scenarios are enabled by default. The API key is required only when `RUN_LIVE_AGENT` is set to `true`.

In [ ]:
import os
import random
import time
from dataclasses import dataclass
from enum import Enum
from typing import Any, Callable, Literal

import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError

RUN_LIVE_AGENT = os.getenv("RUN_LIVE_AGENT", "false").lower() == "true"
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6")

if RUN_LIVE_AGENT and not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "Set OPENAI_API_KEY before enabling the opt-in live agent run."
    )

print(f"Live agent run enabled: {RUN_LIVE_AGENT}")
print(f"Model for live runs: {MODEL}")

## Architecture and responsibility boundaries

The example separates model decisions from infrastructure recovery:

1. The agent decides which business action to request.
2. A typed function tool sends the request to the recovery layer.
3. The recovery layer classifies timeout failures and applies retry, validation, and idempotency rules.
4. A deterministic simulator returns a configured success or failure for each attempt.
5. The agent receives a structured outcome and either answers from confirmed data or escalates.

Keeping retries in application code makes attempt limits and side-effect safety testable instead of asking the model to infer infrastructure policy.

## 1. Define typed tool contracts

Define schemas for order status, escalation requests, confirmed results, recoverable failures, permanent failures, and ambiguous writes. These schemas will provide a stable boundary between the agent and the reliability layer.

In [ ]:
class StrictModel(BaseModel):
    """Reject fields that are not part of the documented tool contract."""

    model_config = ConfigDict(extra="forbid")


class OrderStatus(StrictModel):
    order_id: str = Field(pattern=r"^ORDER-[0-9]{4}$")
    status: Literal["in_transit", "delayed", "delivered"]
    carrier: str
    last_scan: str
    estimated_delivery: str | None = None


class EscalationRequest(StrictModel):
    order_id: str = Field(pattern=r"^ORDER-[0-9]{4}$")
    reason: str = Field(min_length=10)
    idempotency_key: str = Field(min_length=8)


class EscalationRecord(StrictModel):
    escalation_id: str
    order_id: str
    reason: str
    idempotency_key: str
    status: Literal["open"] = "open"


class AttemptEvent(StrictModel):
    operation: str
    attempt: int = Field(ge=1)
    fault_kind: str
    result: str
    error_code: str | None = None
    retryable: bool = False
    side_effect_committed: bool = False


class ToolOutcome(StrictModel):
    status: Literal["success", "failed", "escalation_required"]
    data: dict[str, Any] | None = None
    error_code: str | None = None
    attempts: int = Field(ge=1)
    confirmed_side_effect: bool = False
    events: list[AttemptEvent] = Field(default_factory=list)


class SyntheticToolError(RuntimeError):
    """Base error raised by the synthetic dependency."""

    def __init__(
        self,
        code: str,
        *,
        retryable: bool,
        status_code: int | None = None,
    ) -> None:
        super().__init__(code)
        self.code = code
        self.retryable = retryable
        self.status_code = status_code


class TransientToolError(SyntheticToolError):
    """A failure that may succeed on a later attempt."""


class PermanentToolError(SyntheticToolError):
    """A failure that should not be retried."""


class AcknowledgementLostError(TransientToolError):
    """The write committed, but its success response was lost."""

## 2. Define deterministic fault plans

Each `FaultPlan` contains the result of successive dependency calls. Once its scripted steps are exhausted, it returns `success`. This makes every test reproducible while keeping healthy calls concise.

In [ ]:
class FaultKind(str, Enum):
    SUCCESS = "success"
    TIMEOUT = "timeout"
    RATE_LIMITED = "rate_limited"
    UNAVAILABLE = "unavailable"
    MALFORMED_RESPONSE = "malformed_response"
    INCOMPLETE_RESPONSE = "incomplete_response"
    FORBIDDEN = "forbidden"
    NOT_FOUND = "not_found"
    ACKNOWLEDGEMENT_LOST = "acknowledgement_lost_after_commit"


@dataclass(frozen=True)
class FaultStep:
    kind: FaultKind


@dataclass
class FaultPlan:
    steps: list[FaultStep]
    attempts: int = 0
    last_step: FaultStep | None = None

    def next_step(self) -> FaultStep:
        if self.attempts < len(self.steps):
            step = self.steps[self.attempts]
        else:
            step = FaultStep(FaultKind.SUCCESS)

        self.attempts += 1
        self.last_step = step
        return step


def make_fault_plan(*kinds: FaultKind) -> FaultPlan:
    return FaultPlan([FaultStep(kind) for kind in kinds])

## 3. Create a synthetic delivery service

The in-memory service provides one read operation and one side-effecting operation. Its idempotency ledger stores escalation records by key, so replaying the same request returns the original record rather than creating a duplicate.

In [ ]:
class SyntheticDeliveryService:
    def __init__(self) -> None:
        self.orders = {
            "ORDER-1001": OrderStatus(
                order_id="ORDER-1001",
                status="delayed",
                carrier="Example Carrier",
                last_scan="Regional sorting facility",
                estimated_delivery="2026-07-24",
            )
        }
        self._escalations_by_key: dict[str, EscalationRecord] = {}
        self._escalation_sequence = 0

    @property
    def escalation_count(self) -> int:
        return len(self._escalations_by_key)

    def _raise_precommit_fault(self, kind: FaultKind) -> None:
        if kind == FaultKind.TIMEOUT:
            raise TransientToolError("timeout", retryable=True)
        if kind == FaultKind.RATE_LIMITED:
            raise TransientToolError(
                "rate_limited", retryable=True, status_code=429
            )
        if kind == FaultKind.UNAVAILABLE:
            raise TransientToolError(
                "dependency_unavailable", retryable=True, status_code=503
            )
        if kind == FaultKind.FORBIDDEN:
            raise PermanentToolError(
                "forbidden", retryable=False, status_code=403
            )
        if kind == FaultKind.NOT_FOUND:
            raise PermanentToolError(
                "not_found", retryable=False, status_code=404
            )

    def get_order_status(
        self, order_id: str, fault_plan: FaultPlan
    ) -> dict[str, Any]:
        step = fault_plan.next_step()
        self._raise_precommit_fault(step.kind)

        if step.kind == FaultKind.ACKNOWLEDGEMENT_LOST:
            raise PermanentToolError(
                "invalid_fault_for_read", retryable=False
            )

        order = self.orders.get(order_id)
        if order is None:
            raise PermanentToolError(
                "order_not_found", retryable=False, status_code=404
            )

        if step.kind == FaultKind.MALFORMED_RESPONSE:
            return {"unexpected": "payload"}
        if step.kind == FaultKind.INCOMPLETE_RESPONSE:
            return {"order_id": order_id, "status": order.status}

        return order.model_dump(mode="json")

    def _commit_escalation(
        self, request: EscalationRequest
    ) -> EscalationRecord:
        existing = self._escalations_by_key.get(request.idempotency_key)
        if existing is not None:
            if (
                existing.order_id != request.order_id
                or existing.reason != request.reason
            ):
                raise PermanentToolError(
                    "idempotency_key_conflict", retryable=False
                )
            return existing

        if request.order_id not in self.orders:
            raise PermanentToolError(
                "order_not_found", retryable=False, status_code=404
            )

        self._escalation_sequence += 1
        record = EscalationRecord(
            escalation_id=f"ESC-{self._escalation_sequence:04d}",
            order_id=request.order_id,
            reason=request.reason,
            idempotency_key=request.idempotency_key,
        )
        self._escalations_by_key[request.idempotency_key] = record
        return record

    def create_delivery_escalation(
        self,
        request: EscalationRequest,
        fault_plan: FaultPlan,
    ) -> dict[str, Any]:
        step = fault_plan.next_step()
        self._raise_precommit_fault(step.kind)
        record = self._commit_escalation(request)

        if step.kind == FaultKind.ACKNOWLEDGEMENT_LOST:
            raise AcknowledgementLostError(
                "acknowledgement_lost", retryable=True
            )
        if step.kind == FaultKind.MALFORMED_RESPONSE:
            return {"unexpected": "payload"}
        if step.kind == FaultKind.INCOMPLETE_RESPONSE:
            return {"escalation_id": record.escalation_id}

        return record.model_dump(mode="json")

    def get_escalation_by_key(
        self, idempotency_key: str
    ) -> EscalationRecord | None:
        return self._escalations_by_key.get(idempotency_key)

## 4. Run an unsafe baseline

Call the synthetic service without the recovery layer. This baseline will make the failure modes visible before adding bounded retries, validation, and idempotency controls.

In [ ]:
baseline_service = SyntheticDeliveryService()
baseline_faults = make_fault_plan(FaultKind.TIMEOUT, FaultKind.SUCCESS)

try:
    baseline_service.get_order_status("ORDER-1001", baseline_faults)
except TransientToolError as error:
    print(f"Baseline stopped after one attempt: {error.code}")

assert baseline_faults.attempts == 1

## 5. Implement the recovery policy

Classify simulated timeouts and dependency errors, enforce a bounded retry budget, apply exponential backoff with jitter, validate responses, and return structured escalation results. Permanent errors fail immediately. Ambiguous writes are reconciled against the idempotency ledger before another write is attempted. A later asynchronous tool wrapper will impose the real wall-clock timeout.

In [ ]:
class RecoveryPolicy(StrictModel):
    max_attempts: int = Field(default=3, ge=1, le=10)
    base_delay_seconds: float = Field(default=0.0, ge=0)
    jitter_ratio: float = Field(default=0.0, ge=0, le=1)
    retry_invalid_output: bool = True


class ReliableDeliveryClient:
    def __init__(
        self,
        service: SyntheticDeliveryService,
        policy: RecoveryPolicy,
        *,
        sleep_fn: Callable[[float], None] = time.sleep,
        random_seed: int = 7,
    ) -> None:
        self.service = service
        self.policy = policy
        self.sleep_fn = sleep_fn
        self.random = random.Random(random_seed)

    def _fault_name(self, fault_plan: FaultPlan) -> str:
        if fault_plan.last_step is None:
            return "unknown"
        return fault_plan.last_step.kind.value

    def _wait_before_retry(self, attempt: int) -> None:
        base_delay = self.policy.base_delay_seconds * (2 ** (attempt - 1))
        jitter = self.random.uniform(
            0, base_delay * self.policy.jitter_ratio
        )
        self.sleep_fn(base_delay + jitter)

    def _failure_outcome(
        self,
        *,
        error_code: str,
        attempts: int,
        events: list[AttemptEvent],
    ) -> ToolOutcome:
        return ToolOutcome(
            status="escalation_required",
            error_code=error_code,
            attempts=attempts,
            events=events,
        )

    def get_order_status(
        self, order_id: str, fault_plan: FaultPlan
    ) -> ToolOutcome:
        events: list[AttemptEvent] = []

        for attempt in range(1, self.policy.max_attempts + 1):
            try:
                raw_result = self.service.get_order_status(
                    order_id, fault_plan
                )
                order = OrderStatus.model_validate(raw_result)
            except ValidationError:
                can_retry = (
                    self.policy.retry_invalid_output
                    and attempt < self.policy.max_attempts
                )
                events.append(
                    AttemptEvent(
                        operation="get_order_status",
                        attempt=attempt,
                        fault_kind=self._fault_name(fault_plan),
                        result="invalid_output",
                        error_code="invalid_tool_output",
                        retryable=can_retry,
                    )
                )
                if can_retry:
                    self._wait_before_retry(attempt)
                    continue
                return self._failure_outcome(
                    error_code="invalid_tool_output",
                    attempts=attempt,
                    events=events,
                )
            except SyntheticToolError as error:
                can_retry = (
                    error.retryable and attempt < self.policy.max_attempts
                )
                events.append(
                    AttemptEvent(
                        operation="get_order_status",
                        attempt=attempt,
                        fault_kind=self._fault_name(fault_plan),
                        result="error",
                        error_code=error.code,
                        retryable=can_retry,
                    )
                )
                if can_retry:
                    self._wait_before_retry(attempt)
                    continue
                return self._failure_outcome(
                    error_code=error.code,
                    attempts=attempt,
                    events=events,
                )

            events.append(
                AttemptEvent(
                    operation="get_order_status",
                    attempt=attempt,
                    fault_kind=self._fault_name(fault_plan),
                    result="success",
                )
            )
            return ToolOutcome(
                status="success",
                data=order.model_dump(mode="json"),
                attempts=attempt,
                events=events,
            )

        raise AssertionError("The bounded retry loop returned no outcome.")

    def create_delivery_escalation(
        self,
        request: EscalationRequest,
        fault_plan: FaultPlan,
    ) -> ToolOutcome:
        events: list[AttemptEvent] = []

        for attempt in range(1, self.policy.max_attempts + 1):
            try:
                raw_result = self.service.create_delivery_escalation(
                    request, fault_plan
                )
                record = EscalationRecord.model_validate(raw_result)
            except AcknowledgementLostError as error:
                committed = self.service.get_escalation_by_key(
                    request.idempotency_key
                )
                can_retry = (
                    committed is None
                    and attempt < self.policy.max_attempts
                )
                events.append(
                    AttemptEvent(
                        operation="create_delivery_escalation",
                        attempt=attempt,
                        fault_kind=self._fault_name(fault_plan),
                        result=("reconciled" if committed else "ambiguous"),
                        error_code=error.code,
                        retryable=can_retry,
                        side_effect_committed=committed is not None,
                    )
                )
                if committed is not None:
                    return ToolOutcome(
                        status="success",
                        data=committed.model_dump(mode="json"),
                        attempts=attempt,
                        confirmed_side_effect=True,
                        events=events,
                    )
                if can_retry:
                    self._wait_before_retry(attempt)
                    continue
                return self._failure_outcome(
                    error_code="ambiguous_write",
                    attempts=attempt,
                    events=events,
                )
            except ValidationError:
                committed = self.service.get_escalation_by_key(
                    request.idempotency_key
                )
                can_retry = (
                    committed is None
                    and self.policy.retry_invalid_output
                    and attempt < self.policy.max_attempts
                )
                events.append(
                    AttemptEvent(
                        operation="create_delivery_escalation",
                        attempt=attempt,
                        fault_kind=self._fault_name(fault_plan),
                        result=(
                            "reconciled_invalid_output"
                            if committed
                            else "invalid_output"
                        ),
                        error_code="invalid_tool_output",
                        retryable=can_retry,
                        side_effect_committed=committed is not None,
                    )
                )
                if committed is not None:
                    return ToolOutcome(
                        status="success",
                        data=committed.model_dump(mode="json"),
                        attempts=attempt,
                        confirmed_side_effect=True,
                        events=events,
                    )
                if can_retry:
                    self._wait_before_retry(attempt)
                    continue
                return self._failure_outcome(
                    error_code="invalid_tool_output",
                    attempts=attempt,
                    events=events,
                )
            except SyntheticToolError as error:
                can_retry = (
                    error.retryable and attempt < self.policy.max_attempts
                )
                events.append(
                    AttemptEvent(
                        operation="create_delivery_escalation",
                        attempt=attempt,
                        fault_kind=self._fault_name(fault_plan),
                        result="error",
                        error_code=error.code,
                        retryable=can_retry,
                    )
                )
                if can_retry:
                    self._wait_before_retry(attempt)
                    continue
                return self._failure_outcome(
                    error_code=error.code,
                    attempts=attempt,
                    events=events,
                )

            events.append(
                AttemptEvent(
                    operation="create_delivery_escalation",
                    attempt=attempt,
                    fault_kind=self._fault_name(fault_plan),
                    result="success",
                    side_effect_committed=True,
                )
            )
            return ToolOutcome(
                status="success",
                data=record.model_dump(mode="json"),
                attempts=attempt,
                confirmed_side_effect=True,
                events=events,
            )

        raise AssertionError("The bounded retry loop returned no outcome.")

## 6. Validate the offline recovery core

These smoke scenarios verify the core invariants before a model is involved: healthy reads succeed once, transient reads recover within the retry budget, permanent errors stop immediately, and acknowledgement loss does not create duplicate writes.

In [ ]:
smoke_results: list[dict[str, Any]] = []
policy = RecoveryPolicy(max_attempts=3)

# 1. Healthy read
service = SyntheticDeliveryService()
client = ReliableDeliveryClient(service, policy)
healthy = client.get_order_status(
    "ORDER-1001", make_fault_plan(FaultKind.SUCCESS)
)
assert healthy.status == "success"
assert healthy.attempts == 1
smoke_results.append(
    {"scenario": "healthy_read", "attempts": healthy.attempts, "passed": True}
)

# 2. Timeout followed by success
service = SyntheticDeliveryService()
client = ReliableDeliveryClient(service, policy)
timeout_then_success = client.get_order_status(
    "ORDER-1001",
    make_fault_plan(FaultKind.TIMEOUT, FaultKind.SUCCESS),
)
assert timeout_then_success.status == "success"
assert timeout_then_success.attempts == 2
assert len(timeout_then_success.events) == 2
assert timeout_then_success.events[0].error_code == "timeout"
assert timeout_then_success.events[0].retryable is True
smoke_results.append(
    {
        "scenario": "timeout_then_success",
        "attempts": timeout_then_success.attempts,
        "passed": True,
    }
)

# 3. Permanent authorization failure
service = SyntheticDeliveryService()
client = ReliableDeliveryClient(service, policy)
forbidden_plan = make_fault_plan(FaultKind.FORBIDDEN, FaultKind.SUCCESS)
forbidden = client.get_order_status("ORDER-1001", forbidden_plan)
assert forbidden.status == "escalation_required"
assert forbidden.error_code == "forbidden"
assert forbidden.attempts == 1
assert forbidden_plan.attempts == 1
assert forbidden.events[0].retryable is False
smoke_results.append(
    {"scenario": "permanent_403", "attempts": forbidden.attempts, "passed": True}
)

# 4. Write commits before its acknowledgement is lost
service = SyntheticDeliveryService()
client = ReliableDeliveryClient(service, policy)
request = EscalationRequest(
    order_id="ORDER-1001",
    reason="The delayed shipment needs carrier investigation.",
    idempotency_key="order-1001-carrier-investigation",
)
acknowledgement_lost = client.create_delivery_escalation(
    request,
    make_fault_plan(FaultKind.ACKNOWLEDGEMENT_LOST),
)
assert acknowledgement_lost.status == "success"
assert acknowledgement_lost.confirmed_side_effect is True
assert acknowledgement_lost.attempts == 1
assert acknowledgement_lost.events[0].result == "reconciled"
assert acknowledgement_lost.events[0].side_effect_committed is True
assert service.escalation_count == 1

# Replaying the same business request returns the original record.
replayed = client.create_delivery_escalation(
    request, make_fault_plan(FaultKind.SUCCESS)
)
assert replayed.data == acknowledgement_lost.data
assert service.escalation_count == 1
smoke_results.append(
    {
        "scenario": "ack_lost_without_duplicate",
        "attempts": acknowledgement_lost.attempts,
        "passed": True,
    }
)

offline_smoke_results = pd.DataFrame(smoke_results)
assert offline_smoke_results["passed"].all()
offline_smoke_results

## 7. Expose safe function tools

Wrap the reliability layer with `@function_tool`. Each tool will return a typed outcome containing confirmed data or an explicit escalation reason rather than exposing raw exceptions to the agent.

## 8. Define the agent

Create one agent with instructions that prohibit fabricated tool results and unconfirmed success claims. The agent will use confirmed tool data when available and produce a structured handoff when the recovery policy is exhausted.

## 9. Evaluate the scenario matrix

Run the same workflow against healthy tools, transient failures that recover, exhausted retry budgets, invalid responses, permanent failures, ambiguous writes, and fallback failures. Assertions will check attempt counts, final disposition, output validity, and duplicate side effects.

## 10. Inspect traces and summarize results

Use Agents SDK traces to inspect representative live runs, then render the deterministic scenario results as a table. This follows the recommended workflow of debugging traces first and moving to repeatable datasets and eval runs after the expected behavior is clear.

## Next steps

Adapt the recovery policy to your service-level objectives, API error taxonomy, and idempotency guarantees. Before using the pattern in a sensitive workflow, add domain-specific approval rules, monitoring, and human review.